In [1]:
import os

In [2]:
pwd

'd:\\projects\\wine-quality-mlops\\research'

In [ ]:
os.chdir("..")
%pwd

'd:\\projects\\wine-quality-mlops'

In [1]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    train_path: Path
    test_path: Path

In [2]:
from wine_quality_mlops.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from wine_quality_mlops.utils.io_utils import read_yaml, create_directories

In [3]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        self.artifacts_root = Path(self.config["artifacts_root"])

        create_directories([self.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:

        config = self.config["data_transformation"]

        root_dir = self.artifacts_root / config["root_dir"]

        create_directories([root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=root_dir,
            data_path=Path(config["data_path"]),
            train_path=root_dir / config["train_path"],
            test_path=root_dir / config["test_path"],
        )

        return data_transformation_config    


In [4]:
import os
import sys
from wine_quality_mlops.utils.logger import logger
from wine_quality_mlops.utils.exceptions import (
    CustomException
)
from sklearn.model_selection import train_test_split
import pandas as pd

In [5]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config


    def train_test_splitting(self):

        data = pd.read_csv(self.config.data_path)

        # Split the data into training and test sets
        train, test = train_test_split(
            data,
            test_size=0.25,
            random_state=42
        )

        train.to_csv(self.config.train_path, index=False)

        test.to_csv(self.config.test_path, index=False)

        logger.info("Split data into training and test sets")

        logger.info(f"Train shape: {train.shape}")
        logger.info(f"Test shape: {test.shape}")

        print(train.shape)
        print(test.shape)

In [8]:
os.chdir(r"D:\projects\wine-quality-mlops")

In [9]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_splitting()
except Exception as e:
    raise CustomException(e, sys)

[2026-05-16 15:29:49,957] 22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\config\config.yaml
[2026-05-16 15:29:49,960] 22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\params.yaml
[2026-05-16 15:29:49,961] 22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\schema.yaml
[2026-05-16 15:29:49,963] 40 datascienceLogger - INFO - Created directory: artifacts
[2026-05-16 15:29:49,965] 40 datascienceLogger - INFO - Created directory: artifacts\data_transformation
[2026-05-16 15:29:49,984] 21 datascienceLogger - INFO - Split data into training and test sets
[2026-05-16 15:29:49,984] 23 datascienceLogger - INFO - Train shape: (1199, 12)
[2026-05-16 15:29:49,985] 24 datascienceLogger - INFO - Test shape: (400, 12)
(1199, 12)
(400, 12)


In [7]:
from pathlib import Path

path = Path("artifacts/data_ingestion/raw/winequality-red.csv")

print(path.exists())
print(path.resolve())

False
D:\projects\wine-quality-mlops\research\artifacts\data_ingestion\raw\winequality-red.csv
